# GAN on MNIST — Generating Synthetic Handwritten Digits

This notebook builds, trains, and evaluates a simple DCGAN (Deep Convolutional GAN) on the MNIST dataset.

## 1. Import libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers, Sequential
from tensorflow.keras.datasets import mnist

## 2. Load MNIST dataset

In [ ]:
(X_train, _), (_, _) = mnist.load_data()

print("Dataset shape:", X_train.shape)
print("Image shape:", X_train.shape[1:])

## 3. Preprocess the images

The Generator uses `tanh` as its final activation, so we scale pixel values from `0-255` to `-1-1`.

In [ ]:
X_train = X_train.astype("float32")

X_train = (X_train - 127.5) / 127.5

# Add channel dimension
X_train = np.expand_dims(X_train, axis=-1)

print(X_train.shape)
print(X_train.min(), X_train.max())

## 4. Display real images

In [ ]:
plt.figure(figsize=(8, 8))

for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(X_train[i].squeeze(), cmap="gray")
    plt.axis("off")

plt.suptitle("Real MNIST Images")
plt.show()

## 5. Create TensorFlow dataset

In [ ]:
BATCH_SIZE = 256

dataset = (
    tf.data.Dataset
    .from_tensor_slices(X_train)
    .shuffle(60000)
    .batch(BATCH_SIZE, drop_remainder=True)
)

## 6. Build the Generator

The Generator takes random noise and converts it into a 28 x 28 image.

In [ ]:
LATENT_DIM = 100

generator = Sequential([

    layers.Input(shape=(LATENT_DIM,)),

    # 100 -> 7*7*256
    layers.Dense(7 * 7 * 256, use_bias=False),
    layers.BatchNormalization(),
    layers.LeakyReLU(),

    # 7*7*256 -> 7*7*256
    layers.Reshape((7, 7, 256)),

    # 7x7 -> 14x14
    layers.Conv2DTranspose(
        128,
        kernel_size=5,
        strides=2,
        padding="same",
        use_bias=False
    ),
    layers.BatchNormalization(),
    layers.LeakyReLU(),

    # 14x14 -> 28x28
    layers.Conv2DTranspose(
        64,
        kernel_size=5,
        strides=2,
        padding="same",
        use_bias=False
    ),
    layers.BatchNormalization(),
    layers.LeakyReLU(),

    # Output: 28x28x1
    layers.Conv2DTranspose(
        1,
        kernel_size=5,
        padding="same",
        activation="tanh"
    )
])

generator.summary()

### Test the Generator

In [ ]:
noise = tf.random.normal([16, LATENT_DIM])

generated_images = generator(
    noise,
    training=False
)

print(generated_images.shape)

You should get:

```
(16, 28, 28, 1)
```

At this point the images will look like noise because the Generator hasn't learned yet.

In [ ]:
plt.figure(figsize=(8, 8))

for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(
        generated_images[i, :, :, 0],
        cmap="gray"
    )
    plt.axis("off")

plt.suptitle("Before Training")
plt.show()

## 7. Build the Discriminator

The Discriminator decides whether an image is real or fake.

In [ ]:
discriminator = Sequential([

    layers.Input(shape=(28, 28, 1)),

    layers.Conv2D(
        64,
        kernel_size=5,
        strides=2,
        padding="same"
    ),
    layers.LeakyReLU(),
    layers.Dropout(0.3),

    layers.Conv2D(
        128,
        kernel_size=5,
        strides=2,
        padding="same"
    ),
    layers.LeakyReLU(),
    layers.Dropout(0.3),

    layers.Flatten(),

    layers.Dense(1, activation="sigmoid")
])

discriminator.summary()

## 8. Test the Discriminator

In [ ]:
prediction = discriminator(
    X_train[:5],
    training=False
)

print(prediction.numpy())

The output is a value between 0 and 1. The values are meaningless initially because the Discriminator hasn't been trained.

## 9. Define loss functions

We use Binary Cross Entropy.

In [ ]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(
    from_logits=False
)

### Generator loss

The Generator wants the Discriminator to classify fake images as real (1).

In [ ]:
def generator_loss(fake_output):

    return cross_entropy(
        tf.ones_like(fake_output),
        fake_output
    )

### Discriminator loss

The Discriminator wants:
- Real images -> 1
- Fake images -> 0

In [ ]:
def discriminator_loss(real_output, fake_output):

    real_loss = cross_entropy(
        tf.ones_like(real_output),
        real_output
    )

    fake_loss = cross_entropy(
        tf.zeros_like(fake_output),
        fake_output
    )

    total_loss = real_loss + fake_loss

    return total_loss

## 10. Create optimizers

In [ ]:
generator_optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.0002,
    beta_1=0.5
)

discriminator_optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.0002,
    beta_1=0.5
)

## 11. Create training step

This is the most important part of the GAN.

In [ ]:
@tf.function
def train_step(real_images):

    batch_size = tf.shape(real_images)[0]

    # Generate random noise
    noise = tf.random.normal(
        [batch_size, LATENT_DIM]
    )

    with tf.GradientTape() as gen_tape, \
         tf.GradientTape() as disc_tape:

        # Generate fake images
        fake_images = generator(
            noise,
            training=True
        )

        # Discriminator predictions
        real_output = discriminator(
            real_images,
            training=True
        )

        fake_output = discriminator(
            fake_images,
            training=True
        )

        # Calculate losses
        gen_loss = generator_loss(
            fake_output
        )

        disc_loss = discriminator_loss(
            real_output,
            fake_output
        )

    # Calculate gradients
    gen_gradients = gen_tape.gradient(
        gen_loss,
        generator.trainable_variables
    )

    disc_gradients = disc_tape.gradient(
        disc_loss,
        discriminator.trainable_variables
    )

    # Update Generator
    generator_optimizer.apply_gradients(
        zip(
            gen_gradients,
            generator.trainable_variables
        )
    )

    # Update Discriminator
    discriminator_optimizer.apply_gradients(
        zip(
            disc_gradients,
            discriminator.trainable_variables
        )
    )

    return gen_loss, disc_loss

## 12. Function to display generated images

In [ ]:
NUM_EXAMPLES = 16

seed = tf.random.normal(
    [NUM_EXAMPLES, LATENT_DIM]
)


def show_generated_images(epoch):

    predictions = generator(
        seed,
        training=False
    )

    plt.figure(figsize=(8, 8))

    for i in range(NUM_EXAMPLES):

        plt.subplot(4, 4, i + 1)

        # Convert -1,1 back to 0,1
        image = (predictions[i, :, :, 0] + 1) / 2

        plt.imshow(
            image,
            cmap="gray"
        )

        plt.axis("off")

    plt.suptitle(
        f"Generated Images - Epoch {epoch}"
    )

    plt.show()

## 13. Train the GAN

Start with `EPOCHS = 30` to test that everything works. Once it works, you can increase it to 100 for better generated images.

In [ ]:
EPOCHS = 30

generator_losses = []
discriminator_losses = []

for epoch in range(1, EPOCHS + 1):

    gen_loss_list = []
    disc_loss_list = []

    for real_images in dataset:

        gen_loss, disc_loss = train_step(
            real_images
        )

        gen_loss_list.append(
            float(gen_loss)
        )

        disc_loss_list.append(
            float(disc_loss)
        )

    avg_gen_loss = np.mean(
        gen_loss_list
    )

    avg_disc_loss = np.mean(
        disc_loss_list
    )

    generator_losses.append(
        avg_gen_loss
    )

    discriminator_losses.append(
        avg_disc_loss
    )

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Generator Loss: {avg_gen_loss:.4f} | "
        f"Discriminator Loss: {avg_disc_loss:.4f}"
    )

    if epoch == 1 or epoch % 5 == 0:

        show_generated_images(epoch)

**What you'll see:**
- Initially: random noise
- After several epochs: rough digit shapes
- Later: realistic handwritten digits

## 14. Plot Generator and Discriminator losses

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    generator_losses,
    label="Generator Loss"
)

plt.plot(
    discriminator_losses,
    label="Discriminator Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("GAN Training Loss")

plt.legend()
plt.grid(True)

plt.show()

## 15. Generate final synthetic images

In [ ]:
noise = tf.random.normal(
    [25, LATENT_DIM]
)

synthetic_images = generator(
    noise,
    training=False
)

plt.figure(figsize=(10, 10))

for i in range(25):

    plt.subplot(5, 5, i + 1)

    image = (
        synthetic_images[i, :, :, 0] + 1
    ) / 2

    plt.imshow(
        image,
        cmap="gray"
    )

    plt.axis("off")

plt.suptitle(
    "Synthetic Images Generated by GAN"
)

plt.show()

## 16. Save generated images

In [ ]:
from PIL import Image

os.makedirs(
    "generated_images",
    exist_ok=True
)

for i in range(25):

    image = (
        synthetic_images[i, :, :, 0]
        .numpy()
    )

    # Convert -1,1 -> 0,255
    image = (
        (image + 1) * 127.5
    )

    image = np.clip(
        image,
        0,
        255
    ).astype(np.uint8)

    Image.fromarray(
        image
    ).save(
        f"generated_images/image_{i+1}.png"
    )

print("Images saved successfully!")

Your folder will look like:

```
GAN_Project/
|
|-- GAN_MNIST.ipynb
|
+-- generated_images/
    |-- image_1.png
    |-- image_2.png
    |-- image_3.png
    |-- ...
    +-- image_25.png
```

## 17. Save the trained models

In [ ]:
generator.save(
    "generator_mnist.keras"
)

discriminator.save(
    "discriminator_mnist.keras"
)

print("Models saved successfully!")

## Complete GAN architecture

```
                  RANDOM NOISE
                     |
                     v
              +-------------+
              |  GENERATOR  |
              +------+------+
                     |
                     v
              SYNTHETIC IMAGE
                     |
             +-------+--------+
             |                |
             v                v
        REAL IMAGE        FAKE IMAGE
             |                |
             +-------+--------+
                     |
                     v
             +-------------+
             |DISCRIMINATOR|
             +------+------+
                    |
                    v
               REAL / FAKE
```

**Note:** For your practical, run this first with `EPOCHS = 30`. Once it works, change it to `100` for better generated images.